In [ ]:
from langchain_ollama import OllamaEmbeddings, OllamaLLM
from langchain.prompts import ChatPromptTemplate

import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F

# These classes are assumed to be defined in your environment
from ChromaVDB.chroma import ChromaFramework
from DeepGraphDB import DeepGraphDB

gdb = DeepGraphDB()
gdb.load_graph("/home/cc/PHD/dglframework/DeepKG/DeepGraphDB/graphs/primekg.bin")
vdb = ChromaFramework(persist_directory="./ChromaVDB/chroma_db")

records = vdb.list_records()

names = [record['name'] for record in records if record['embedding_type'] == 'graph']
"alibayram/medgemma:27b"
"35884" # Diffuse b-cell lymphoma in graph

In [ ]:
import json
from typing import List, Dict, Tuple, Any

from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_core.pydantic_v1 import BaseModel, Field

# Define the output schema for LangChain's PydanticOutputParser
class ScoredEntity(BaseModel):
    """An entity from the biological knowledge graph with an assigned importance score."""
    entity: str = Field(description="The entity in 'entity_type: name' format (e.g., disease: B-cell lymphoma)")
    score: int = Field(description="Importance score from 2 to 5, relative to the source entity. 1 is default for unlisted entities.")

class EntityScoreOutput(BaseModel):
    """List of important entities and their scores."""
    important_entities: List[ScoredEntity] = Field(
        description="A list of entities from the subgraph with an importance score greater than 1."
    )

# Helper function to escape curly braces in a string
def escape_curly_braces(text: str) -> str:
    """Escapes single curly braces to double curly braces for f-string compatibility."""
    # Replace { with {{ and } with }}
    return text.replace("{", "{{").replace("}", "}}")

def score_subgraph_entities(
    source_entity: str,
    k_hops: int,
    # subgraph_triplets: List[Tuple[str, str, str]],
    subgraph_triplets: List[str],
    meta_paths: List[str],
    ollama_model_name: str = "alibayram/medgemma:27b"
) -> List[Dict[str, Any]]:
    """
    Uses MedGemma via Ollama and LangChain to score entities in a k-hop subgraph.

    Args:
        source_entity (str): The source entity (e.g., "disease: B-cell lymphoma").
        k_hops (int): The number of hops for the subgraph extraction.
        subgraph_triplets (List[Tuple[str, str, str]]): List of (head, relation, tail) triplets.
        meta_paths (List[str]): List of descriptions of meta-paths in the subgraph.
        ollama_model_name (str): The name of the Ollama model to use (e.g., "medgemma:27b").

    Returns:
        List[Dict[str, Any]]: A list of dictionaries, each with 'entity' and 'score' for important entities.
    """

    # Initialize the Ollama LLM with structured output directly
    llm = ChatOllama(model=ollama_model_name).with_structured_output(EntityScoreOutput)

    # # Convert triplets to a string format suitable for the prompt
    # triplets_str = "\n".join([f'("{h}", "{r}", "{t}")' for h, r, t in subgraph_triplets])
    # if not triplets_str:
    #     triplets_str = "No triplets found in the subgraph."

    # # Convert meta-paths to a string format
    # meta_paths_str = "\n".join([f'- "{mp}"' for mp in meta_paths])
    # if not meta_paths_str:
    #     meta_paths_str = "Not applicable, k=1."
    
    # Define the prompt template
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "You are an expert in biological knowledge graphs and entity relationship analysis. "
                "Your task is to identify and score the most important entities within a k-hop subgraph, "
                "relative to a given source entity. The importance score should reflect how strongly and "
                "directly an entity is related to the source entity, and its potential biological "
                "significance in that context. \n"
                "The entities are provided in the format entity_type: entity_name "
                "\n\n"
                "**Instructions for Scoring:**\n"
                "1. **Analyze Relationships:** Carefully examine the provided entities. Also consider how meta-paths highlight indirect but potentially significant connections.\n"
                "2. **Identify Importance:** Focus on entities that have a direct and strong biological connection to the Source Entity, or those that are central to multiple important relationships within the subgraph. Entities directly connected in the first hop are generally more important.\n"
                "3. **Scoring Scale (1-5):**\n"
                "   * **Score 5 (Highly Important):** Entities directly and strongly related to the Source Entity, representing core biological associations. These are often primary targets, key regulators, or direct causes/treatments.\n"
                "   * **Score 4 (Very Important):** Entities with strong, direct, and highly relevant connections, or those central to significant meta-paths from the Source Entity.\n"
                "   * **Score 3 (Moderately Important):** Entities with clear, but perhaps less direct or less universally critical connections. They still offer valuable biological insights.\n"
                "   * **Score 2 (Slightly Important):** Entities with indirect or less prominent connections, but still part of the subgraph and potentially relevant in a broader context.\n"
                "   * **Score 1 (Least Important / Baseline):** All other entities within the subgraph that are not explicitly assigned a higher score, or entities whose relevance is minimal in the context of the Source Entity.\n\n"
                "**Output Format:**\n"
                "Provide your response as a JSON object, specifically as a list under the key 'important_entities'. "
                "Each item in the list should be a dictionary containing 'entity' (in 'entity_type: name' format) "
                "and its assigned 'score'. Only include entities with a score greater than 1. All entities not "
                "listed explicitly in your output are implicitly considered to have a score of 1.\n"
                # "Provide your response as a JSON object."
                # "Each item in the list should be a dictionary containing 'entity'"
                # "and its assigned 'score' (eg: 'tk53': 4). Only include entities with a score greater than 1. All entities not "
                # "listed explicitly in your output are implicitly considered to have a score of 1.\n"
            ),
            ("human",
             "**Source Entity:**\n{source_entity}\n\n"
             "**Subgraph Entities:**\n{subgraph_entities}\n\n"
             "**Meta-Paths:**\n{meta_paths}\n\n"
             "**Begin your analysis.**"
            ),
        ]
    )

    # Create the LangChain chain
    chain = prompt | llm

    # Invoke the chain with the subgraph data
    response = chain.invoke({
        "source_entity": source_entity,
        "subgraph_entities": subgraph_triplets,
        "meta_paths": meta_paths
    })

    print(response)
    
    # 'response' is an instance of EntityScoreOutput
    # You return response.important_entities, which is a List[ScoredEntity]
    return response.important_entities


In [ ]:
# # Example 1: 1-hop subgraph
# source_entity_1 = "disease: B-cell lymphoma"
# k_hops_1 = 1
# triplets_1, meta_paths_1 = get_k_hop_subgraph_mock(source_entity_1, k=k_hops_1)

# print(f"\n--- Scoring for {source_entity_1} ({k_hops_1}-hop) ---")
# scored_entities_1 = score_subgraph_entities(source_entity_1, k_hops_1, triplets_1, meta_paths_1)
# # FIX: Use .dict() instead of .model_dump() for Pydantic v1.x compatibility
# json_serializable_entities_1 = [entity.dict() for entity in scored_entities_1]
# print(json.dumps(json_serializable_entities_1, indent=2))

# # Example 2: 2-hop subgraph
# source_entity_2 = "disease: B-cell lymphoma"
# k_hops_2 = 2
# triplets_2, meta_paths_2 = get_k_hop_subgraph_mock(source_entity_2, k=k_hops_2)

# print(f"\n--- Scoring for {source_entity_2} ({k_hops_2}-hop) ---")
# scored_entities_2 = score_subgraph_entities(source_entity_2, k_hops_2, triplets_2, meta_paths_2)
# # FIX: Use .dict() instead of .model_dump()
# json_serializable_entities_2 = [entity.dict() for entity in scored_entities_2]
# print(json.dumps(json_serializable_entities_2, indent=2))

# # Example 3: Entity with no predefined subgraph
# source_entity_3 = "gene: TP53"
# k_hops_3 = 1
# triplets_3, meta_paths_3 = get_k_hop_subgraph_mock(source_entity_3, k=k_hops_3)

# print(f"\n--- Scoring for {source_entity_3} ({k_hops_3}-hop) ---")
# scored_entities_3 = score_subgraph_entities(source_entity_3, k_hops_3, triplets_3, meta_paths_3)
# # FIX: Use .dict() instead of .model_dump()
# json_serializable_entities_3 = [entity.dict() for entity in scored_entities_3]
# print(json.dumps(json_serializable_entities_3, indent=2))

In [ ]:
def get_k_hop_subgraph(source_id: int, k: int) -> Tuple[List[str], List[str]]:
    sub_g = gdb.get_k_hop_neighbors(seed_nodes=[source_id], k=k)

    g_list = []
    for i in range(k):
        g_list.extend(list(set(sub_g[i+1])))

    entities = []
    for gid in g_list:
        entity, lid = gdb.global_to_local_mapping[gid]
        entities.append(entity+": "+gdb.node_data[entity]['name'][lid])

    relation_steps = []

    for i in range(k+1):
        e_types = []
        for node_id in list(sub_g[i]):
            if gdb.global_to_local_mapping[node_id][0] not in e_types:
                e_types.append(gdb.global_to_local_mapping[node_id][0])
        relation_steps.append(e_types)
        e_types = []

    meta_paths = []

    for i in range(len(relation_steps)-1):
        meta_path = []
        for ctype in gdb.graph.canonical_etypes:
            if ctype[0] in relation_steps[i] and ctype[2] in relation_steps[i+1]:
                meta_path.append(ctype)
        if meta_path:
            meta_paths.append(meta_path)

    combined_mp = []

    for i in range(len(meta_paths)-1):
        for start_ctype in meta_paths[i]:
            # source_entities = list(set(m[0] for m in meta_paths[i]))
            for ctype in meta_paths[i+1]:
                if ctype[0] == start_ctype[2]:
                    combined_mp.append(str(start_ctype) + "->" + str(ctype))

    return entities, combined_mp

In [ ]:
source_entity_2 = "disease: B-cell lymphoma"
source_entity_gid = 35884
k_hops_2 = 1
triplets_2, meta_paths_2 = get_k_hop_subgraph(source_entity_gid, k=k_hops_2)

print(f"\n--- Scoring for {source_entity_2} ({k_hops_2}-hop) ---")
scored_entities_2 = score_subgraph_entities(source_entity_2, k_hops_2, triplets_2, meta_paths_2)
# FIX: Use .dict() instead of .model_dump()
json_serializable_entities_2 = [entity.dict() for entity in scored_entities_2]
print(json.dumps(json_serializable_entities_2, indent=2))

In [ ]:
scored_entities_2